# Semana 4: Apuntes de la clase

**Valor terminal, enterprise value y análisis de sensibilidad** (CFA L2, Equity: *Free Cash Flow Valuation*)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/04_valor_terminal_ev/clase04_apuntes.ipynb)

Este notebook resume los conceptos que debiste llevarte de la clase, con los ejemplos numéricos ejecutables. Úsalo para repasar antes de la tarea y del TR1.

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas que usamos esta semana; las de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| VT | Valor Terminal | Valor de todos los flujos posteriores al horizonte de proyección; suele ser 60 a 80 por ciento del valor total. |
| EV | *Enterprise Value* (valor de la firma) | Lo que vale el negocio operativo completo: capitalización mas deuda mas preferentes mas minoritarios menos caja. |
| EBITDA | *Earnings Before Interest, Taxes, Depreciation and Amortization* | Resultado operativo antes de depreciación; la métrica favorita de los múltiplos de salida. |
| EV/EBITDA | Múltiplo de valor de la firma sobre EBITDA | El múltiplo estándar para el método del múltiplo de salida y para comparables (semana 5). |
| g | Tasa de crecimiento perpetuo | El crecimiento de largo plazo del flujo; nunca mayor que el crecimiento nominal de la economía. |
| RR | Tasa de reinversión (*reinvestment rate*) | Fracción del resultado operativo que se reinvierte; el crecimiento sale de aquí: g = RR x ROIC. |
| ROIC | *Return on Invested Capital* | Cuánto rinde el capital invertido; en la madurez tiende a converger al WACC. |
| PS | *Preferred Stock* (acciones preferentes) | Financistas híbridos: entran en el EV y se restan en el puente al equity común. |
| MI | *Minority Interest* (interés minoritario) | La parte de subsidiarias consolidadas que no es de la matriz; también cruza el puente. |
| FCFF | *Free Cash Flow to the Firm* | El flujo que se descuenta al WACC para llegar al EV (semana 3). |
| WACC | *Weighted Average Cost of Capital* | La tasa de descuento del FCFF (semana 2). |
| DCF | *Discounted Cash Flow* | El método completo que hoy terminamos de armar. |


## 1. El valor terminal domina el DCF

En un DCF de dos etapas, el VT concentra entre 60 y 80 por ciento del valor total (en el ejemplo de la clase: 79 por ciento). Conclusión: los supuestos del VT son los supuestos de la valoración, y merecen más escrutinio que ningún otro.

## 2. Método 1: perpetuidad creciente (Gordon)

$$VT_n = \frac{\text{FCFF}_{n+1}}{\text{WACC} - g} = \frac{\text{FCFF}_n (1+g)}{\text{WACC} - g}$$

Dos errores mecánicos que regalan puntos en los exámenes: usar el flujo del año $n$ en lugar del $n+1$ (subestima el VT en el factor $1+g$), y olvidar descontar el VT a valor presente (queda expresado en soles del año $n$).

La disciplina del $g$: nunca mayor que el crecimiento nominal de largo plazo de la economía (2 a 4 por ciento para una empresa en dólares). Y la consistencia con la reinversión:

$$g = RR \times ROIC$$

crecer no es gratis; si el $g$ proyectado no está financiado con la reinversión del flujo, el modelo se contradice.

## 3. Método 2: múltiplo de salida

$$VT_n = \text{múltiplo} \times \text{EBITDA}_n$$

Ancla el VT a precios de mercado observables, al costo de meter valoración relativa en un modelo intrínseco. El control cruzado entre métodos es la práctica profesional: del Gordon siempre se extrae el múltiplo implícito ($VT_n / \text{EBITDA}_n$) y se compara contra los comparables del sector.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils.finanzas import valor_terminal, valor_terminal_multiplo, dcf_dos_etapas

# Ejemplo de la clase: FCFF_0 = 50.5, crecimiento 6, 5.5, 5, 4.5, 4 por ciento
WACC, g_perp, D, acciones = 0.0914, 0.04, 300, 100
proy = []
f = 50.5
for g in [0.06, 0.055, 0.05, 0.045, 0.04]:
    f *= (1 + g)
    proy.append(f)
print("FCFF proyectado:", [round(x, 1) for x in proy])

VT = valor_terminal(proy[-1], WACC, g_perp)
res = dcf_dos_etapas(proy, WACC, VT)
print(f"VT (anio 5)   = {VT:,.1f}")
print(f"EV            = {res['valor']:,.1f}  (VP flujos {res['vp_flujos']:,.1f} + VP del VT {res['vp_vt']:,.1f})")
print(f"Peso del VT   = {res['peso_vt']:.0%}")
print(f"Equity        = {res['valor'] - D:,.1f}  |  por accion = {(res['valor'] - D)/acciones:,.2f}")

Salida esperada: VT = 1,303.9, EV = 1,069.4 (79 por ciento es VT), equity = 769.4 y 7.69 por acción.

## 4. El control cruzado en acción

In [ ]:
EBITDA_0 = 130          # EBIT 100 + depreciacion 30
e = EBITDA_0
for g in [0.06, 0.055, 0.05, 0.045, 0.04]:
    e *= (1 + g)

multiplo_implicito = VT / e
print(f"EBITDA del anio 5 = {e:,.1f}")
print(f"Multiplo implicito del Gordon = {multiplo_implicito:.1f}x")

# Y al reves: VT por multiplo de salida de 8x
VT_m = valor_terminal_multiplo(e, 8)
res_m = dcf_dos_etapas(proy, WACC, VT_m)
print(f"EV con multiplo de salida 8x  = {res_m['valor']:,.0f}  (Gordon daba {res['valor']:,.0f})")

El Gordon implica 7.9x EBITDA. Si las comparables maduras del sector cotizan entre 6x y 9x, el supuesto es defendible; si implicara 18x, habría que revisar $g$ o WACC. Los dos métodos difieren aquí en 1.4 por ciento: se validan mutuamente.

### ¿El g está financiado? El control de la reinversión

La relación $g = RR \times ROIC$ también sirve de control. Con los datos de la empresa del ejemplo (EBIT 100, t 29.5 por ciento, depreciación 30, capex 40, incremento de capital de trabajo 10) se puede extraer el ROIC que el modelo está suponiendo a perpetuidad.

In [ ]:
# Reinversion neta = capex - depreciacion + incremento de capital de trabajo
nopat = 100 * (1 - 0.295)
reinversion = 40 - 30 + 10
RR = reinversion / nopat
ROIC_implicito = g_perp / RR
print(f"NOPAT = {nopat:.1f} | reinversion neta = {reinversion} | RR = {RR:.1%}")
print(f"ROIC implicito en g = {g_perp:.0%}: {ROIC_implicito:.1%}  (WACC = {WACC:.2%})")

Salida esperada: RR = 28.4 por ciento y ROIC implícito = 14.1 por ciento. El modelo supone que la empresa gana unos 5 puntos por encima de su costo de capital para siempre. No es imposible, pero es un supuesto fuerte que hay que poder defender (una ventaja competitiva durable); si el ROIC convergiera al WACC, sostener el mismo $g$ exigiría reinvertir mucho más y el flujo libre sería menor.

## 5. Enterprise value y el puente completo

$$EV = E + D + PS + MI - C$$

La caja se resta porque al comprar la empresa te la llevas (y porque no es un activo operativo que el FCFF remunere). El puente de vuelta: del EV se resta la deuda a valor de mercado, los preferentes y los minoritarios, se suma la caja excedente, y se divide entre acciones diluidas. Las tres finuras: deuda a mercado, caja operativa vs excedente, y dilución.

## 6. Sensibilidad: el valor es un rango

In [ ]:
import pandas as pd

tabla = pd.DataFrame(index=[0.0814, 0.0914, 0.1014], columns=[0.03, 0.04, 0.05], dtype=float)
for w in tabla.index:
    for gg in tabla.columns:
        vt = valor_terminal(proy[-1], w, gg)
        tabla.loc[w, gg] = dcf_dos_etapas(proy, w, vt)["valor"]

tabla.index.name = "WACC"
tabla.columns.name = "g perpetuo"
tabla.round(0)

Un punto de WACC y un punto de $g$, ambos defendibles, mueven el EV de 795 a 1,691: más del doble. Por eso un informe profesional reporta un rango con supuestos explícitos, no un número puntual. Así se pedirá en el TR1.

## 7. Respuestas a los ítems de la clase

**1. B.** $\text{FCFF}_5/(\text{WACC}-g)$ usa el flujo del año 5 en lugar del 6: subestima el VT exactamente en el factor $(1+g)$.

**2. A.** $EV = 500 + 200 - 50 = 650$. La caja se resta.

**3. B.** Un múltiplo implícito de 18x contra comparables a 8x dice que la perpetuidad es demasiado optimista: revisar $g$ o WACC. Promediar dos métodos inconsistentes (C) no arregla el error de supuestos.

**4. B.** El producto de un DCF es un rango con sensibilidad sobre los supuestos críticos; el punto único esconde la incertidumbre del modelo.

## 8. Lista de verificación

Sabes de esta semana si puedes: calcular el VT por los dos métodos y explicar cuándo prefieres cada uno; extraer el múltiplo implícito y usarlo como control; escribir la definición completa del EV y cruzar el puente con sus tres finuras; construir una tabla de sensibilidad y explicar por qué el rango es el resultado, no el punto.